# 🏦 Loan Approval Prediction with Machine Learning
**Author:** ML Engineering Team
**Dataset:** Financial Institution Historical Loan Records
**Goal:** Build an end-to-end Machine Learning pipeline to predict loan approvals, analyze creditworthiness factors, and provide explainable risk assessment.

## 1. Import Essential Libraries
We import data manipulation, visualization, preprocessing, and machine learning model libraries.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import joblib

# Configure visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['font.size'] = 11

## 2. Load the Dataset
We load `dataset/loan_data.csv` and inspect its shape and top records.

In [2]:
df = pd.read_csv('../dataset/loan_data.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Data Overview & Missing Values Inspection
Let's examine column data types, missing values, and statistical summaries.

In [3]:
df.info()

print("\n--- Missing Value Count ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [4]:
# Summary statistics for numerical variables
df.describe()

## 4. Exploratory Data Analysis (EDA)
Here we analyze target distribution and key bivariate relationships.

In [5]:
# 4.1 Target Variable Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Loan_Status', hue='Loan_Status', palette={'Y': '#22c55e', 'N': '#ef4444'}, legend=False)
plt.title('Loan Approval Distribution (Target)', fontweight='bold')
plt.xlabel('Loan Status (Y: Approved, N: Rejected)')
plt.ylabel('Application Count')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width()/2, p.get_height()/2), ha='center', color='white', fontweight='bold')
plt.show()

In [6]:
# 4.2 Credit History vs Loan Status
plt.figure(figsize=(7, 4))
sns.countplot(data=df.dropna(subset=['Credit_History']), x='Credit_History', hue='Loan_Status', palette=['#ef4444', '#22c55e'])
plt.title('Credit History vs Loan Approval (Decisive Factor)', fontweight='bold')
plt.xlabel('Credit History (0 = Poor/Default, 1 = Meets Guidelines)')
plt.ylabel('Applications')
plt.legend(title='Loan Status', labels=['Rejected (N)', 'Approved (Y)'])
plt.show()

In [7]:
# 4.3 Property Area & Education vs Approval
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(ax=axes[0], data=df, x='Property_Area', hue='Loan_Status', palette=['#22c55e', '#ef4444'])
axes[0].set_title('Property Area vs Loan Approval', fontweight='bold')

sns.countplot(ax=axes[1], data=df, x='Education', hue='Loan_Status', palette=['#22c55e', '#ef4444'])
axes[1].set_title('Education Level vs Loan Approval', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Cleaning & Feature Preparation
We clean strings (e.g. `Dependents` '3+' -> 3), define feature matrices `X` and target `y`.

In [8]:
df_clean = df.copy()
df_clean['Dependents'] = df_clean['Dependents'].astype(str).str.replace('+', '', regex=False)
df_clean['Dependents'] = df_clean['Dependents'].replace('nan', np.nan)

# Encode target: Y -> 1, N -> 0
df_clean['Loan_Status'] = df_clean['Loan_Status'].map({'Y': 1, 'N': 0})

cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

X = df_clean[cat_cols + num_cols]
y = df_clean['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

## 6. Preprocessing Pipelines with Scikit-Learn
We build a leak-free `ColumnTransformer`:
- Numerical: Median imputation + `StandardScaler`
- Categorical: Most frequent imputation + `OneHotEncoder`

In [9]:
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

## 7. Model Training & Comparison
We train multiple classifiers and perform 5-fold cross-validation.

In [10]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=250, max_depth=6, min_samples_split=4, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, max_depth=3, learning_rate=0.05, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=5, random_state=42)
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

trained_pipelines = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'CV Acc Mean (%)': round(cv_scores.mean() * 100, 2),
        'Test Acc (%)': round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'ROC-AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2)
    })

res_df = pd.DataFrame(results).sort_values(by='Test Acc (%)', ascending=False)
res_df

## 8. Detailed Evaluation of Random Forest
We inspect the confusion matrix, classification report, and ROC curve.

In [11]:
rf_pipe = trained_pipelines['Random Forest']
y_pred_rf = rf_pipe.predict(X_test)
y_prob_rf = rf_pipe.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Rejected', 'Approved']))

cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Rejected', 'Pred Approved'], yticklabels=['Actual Rejected', 'Actual Approved'])
plt.title('Random Forest Confusion Matrix', fontweight='bold')
plt.show()

## 9. Feature Importance Analysis
Extracting tree feature importances helps understand model decisions.

In [12]:
encoded_cat_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols)
all_feature_names = list(num_cols) + list(encoded_cat_names)
rf_clf = rf_pipe.named_steps['classifier']

feat_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance (%)': rf_clf.feature_importances_ * 100
}).sort_values(by='Importance (%)', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_df.head(8), x='Importance (%)', y='Feature', palette='viridis')
plt.title('Top Factors Influencing Loan Approval', fontweight='bold')
plt.show()

## 10. Real-Time Inference Demonstration
Let's test an arbitrary applicant sample through the pipeline.

In [13]:
sample_applicant = pd.DataFrame([{
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': '2',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 6000,
    'CoapplicantIncome': 2000.0,
    'LoanAmount': 150.0,
    'Loan_Amount_Term': 360.0,
    'Credit_History': 1.0,
    'Property_Area': 'Semiurban'
}])

pred = rf_pipe.predict(sample_applicant)[0]
prob = rf_pipe.predict_proba(sample_applicant)[0]

status_str = '🟢 APPROVED' if pred == 1 else '🔴 NOT APPROVED'
print(f"Prediction: {status_str}")
print(f"Confidence: {prob[pred]*100:.2f}%")
print(f"Approval Probability: {prob[1]*100:.2f}% | Rejection Probability: {prob[0]*100:.2f}%")

## 11. Conclusion
1. **Credit History** is the single most vital factor determining loan approval.
2. **Semiurban** property areas and **Graduate** status show positively correlated approval likelihoods.
3. Ensembles (**Random Forest** and **Extra Trees**) and **Logistic Regression** achieve robust test accuracy (>85%) and high recall (>98%).
4. The serialized pipeline in `model/loan_model.pkl` is directly deployable into our interactive web application.